# Insole2Kn3D — distributed plantar vGRF to center of pressure

**Article:** Sakib Mahmud, Muhammad E. H. Chowdhury, and Faycal Bensaali, *3D Foot Kinetics Estimation From Distributed VGRF From Smart Insoles via 1D Domain Transformation*. **IEEE JBHI 30(3), 2656–2667 (2026)**. [DOI: 10.1109/JBHI.2025.3605296](https://doi.org/10.1109/JBHI.2025.3605296).

The 16 pressure channels from the configured insole foot are inputs. Select `CoPx` (anteroposterior) or `CoPz` (mediolateral). The default reference is the instrumented treadmill. A missing treadmill CoP channel stops loading; an insole-reference diagnostic must be explicitly configured and labeled.

**Start here:** run cells in order using the project environment. The default is a small synthetic demonstration with training disabled. For real experiments, supply the dataset and change the two visible settings below.

The maintained implementation uses participant-disjoint validation and explicit scale/provenance checks. It is **not certified to reproduce either article's numerical results**. See the [repository guide](README.md), [paper correspondence and citation](docs/PAPERS.md), and [data contract](docs/DATA_CONTRACT.md).

## 1. Load the shared workflow

Install `requirements-notebooks.txt`, open this notebook from the repository folder, and select that environment as the kernel. Restart the kernel after changing installed packages. All three notebooks call the same package as the command line, so fixes and metric definitions stay consistent.

In [ ]:
from pathlib import Path
import json
import sys
from dataclasses import replace

# Locate the checkout when Jupyter starts in the repository or a subdirectory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "kinetics").is_dir() and (p / "configs").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the repository folder; see README.md.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from kinetics.config import ExperimentConfig
from kinetics.data import load_dataset, synthetic_dataset
from kinetics.runner import inspect_dataset, demo_configuration, run_experiment
from kinetics.preprocessing import prepare_fold

print("Shared workflow loaded from:", ROOT)

## 2. Choose demo or research data

`USE_DEMO_DATA=True` supplies deterministic synthetic waveforms, a small UNet, one fold and one epoch. These signals check software only. `RUN_TRAINING=False` lets you inspect the full workflow before running optimization. Set it to `True` when ready.

For research data, set `USE_DEMO_DATA=False`, edit the JSON file below, and keep its target, foot, units, reference source and input features consistent. Paths in `configs/` are relative to the repository root. A JSON elsewhere uses its own directory unless `project_dir` overrides it. To override one setting interactively, use `replace(config, target="Mx")`; validate changes before training.

In [ ]:
USE_DEMO_DATA = True
RUN_TRAINING = False
CONFIG_FILE = ROOT / "configs" / "insole_cop.json"

config = ExperimentConfig.from_json(CONFIG_FILE)
if USE_DEMO_DATA:
    config = demo_configuration(config)
    print("SYNTHETIC DEMONSTRATION — software behavior only")

print(json.dumps({key: config.to_dict()[key] for key in (
    "study", "target", "side", "model", "data_format", "dataset_state",
    "normalization", "target_units", "target_source", "window_length",
    "sample_rate", "train_stride", "fold_ids", "epochs", "output_dir"
)}, indent=2))

## 3. Load aligned inputs and references

The loader requires `X=(record,time,channel)` and `Y=(record,time,1)`, finite values, unique record identifiers and one participant/speed/mode label per record. MATLAB exports must be v7.3 HDF5 files. The loader selects named channels and never substitutes another reference source.

If real files are missing, stop and follow the [data preparation instructions](docs/DATA_CONTRACT.md). Keep source files unchanged. Do not replace missing measurements with zeros or declare normalized legacy targets to have physical units.

In [ ]:
data = synthetic_dataset(config) if USE_DEMO_DATA else load_dataset(config)
inspection = inspect_dataset(config, data)
print("Input:", data.x.shape, "| target:", data.y.shape)
print("Participants:", inspection["participants"])
print("Input channels:", ", ".join(data.feature_names))
print("Reference:", data.source, "| units:", data.units)
print("Records longer than the requested window:", inspection["augmentation_available"])

## 4. Review participant splits and training transforms

Outer test folds are P11/P12, P09/P10, P07/P08, P05/P06 and P04. A validation participant is selected only from the outer training pool. Feature selection and optional min–max transforms are fitted only on training participants. Test participants never select checkpoints or preprocessing. This differs from the articles' stated random 10% training-sample validation.

Windows stay inside individual source records. `train_stride` controls training overlap; validation/test windows do not overlap. Existing four-second windows cannot recover the continuous data needed for 80% overlap. For source-native data, predictions are restored using the saved training transform. Legacy data remain in their stored scale. **NRMSE = source-scale RMSE / training target range**, regardless of whether normalization is enabled; it is undefined if that range is zero.

Incomplete trailing windows are dropped; the split manifest records discarded tail samples. Test references remain in their exact source scale for evaluation.

In [ ]:
for item in inspection["folds"]:
    print(f"Fold {item['fold']}: {item['subjects']} | windows {item['planned_window_counts']}")

# Preview one fold. The runner prepares each requested fold independently.
train, validation, test, inverse_target, split_manifest = prepare_fold(data, config, config.fold_ids[0])
print("Selected features:", split_manifest["features"])
print("NRMSE denominator:", split_manifest["nrmse_denominator"])
print("Training windows:", train.x.shape, "| validation:", validation.x.shape, "| test:", test.x.shape)

## 5. Inspect one aligned training window

Inputs and targets have different meanings and may have different units. Separate axes show the selected input and the reference on the model scale. Look for alignment, constant signals and implausible ranges before training. This figure contains source signals, not model predictions.

In [ ]:
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
import numpy as np

time = np.arange(config.window_length) / config.sample_rate
figure, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, constrained_layout=True)
axes[0].plot(time, train.x[0, :, 0], color="#1f4e79")
axes[0].set(ylabel="Input / model scale", title=train.feature_names[0])
axes[1].plot(time, train.y[0, :, 0], color="#bc4b28")
axes[1].set(xlabel="Time (s)", ylabel="Target / model scale",
            title=f"{config.side} {config.target} | {data.source}")
if USE_DEMO_DATA:
    figure.suptitle("Synthetic demonstration only")
plt.show()
plt.close(figure)

## 6. Verify model inputs and named outputs

The registry in [modeling.py](kinetics/modeling.py) selects an implemented architecture. `out` is always the full-resolution prediction. Optional auxiliary heads receive averaged targets at their actual resolutions. A forward pass checks dimensions; it does not establish equivalence to Km2KnNet or Ke2KeNet. See the [model guide](models/README.md).

In [ ]:
import tensorflow as tf
from kinetics.modeling import build_model, prepare_targets

tf.keras.utils.set_random_seed(config.seed)
model = build_model(config, int(train.x.shape[2]))
head_targets = prepare_targets(model, train.y[:1])
probe = model(train.x[:1], training=False)
assert all(tuple(probe[name].shape) == head_targets[name].shape for name in head_targets)
print("TensorFlow:", tf.__version__, "| parameters:", model.count_params())
print("Output shapes:", {name: tuple(value.shape) for name, value in probe.items()})
# Release the preview model; training creates a fresh model for each fold.
del model, probe, head_targets
tf.keras.backend.clear_session()

## 7. Train and evaluate the selected folds

Return to Section 2 and set `RUN_TRAINING=True` to enable optimization. For a first real-data run, choose one fold and one epoch before committing to a full experiment. Early stopping and learning-rate scheduling monitor the total validation loss (primary weight 1, auxiliary weight 0.5 each). The best checkpoint is restored before test prediction.

Every run has a unique directory. It saves configuration, versions, code/file and loaded-array hashes, split manifests, checkpoints, history and predictions. Failure or interruption records the current fold and completed folds. A fresh invocation starts a new run; automatic training resume is not implemented.

In [ ]:
run_dir = None
if RUN_TRAINING:
    run_dir = run_experiment(config, data)
else:
    print("Training disabled. Review Sections 2–6, then enable RUN_TRAINING in Section 2.")

## 8. Review metrics and a predicted window

MAE and RMSE use the stored source scale after reversing an optional training transform. NRMSE uses the training source range. Signed predictions are preserved. PCC, R², CCC and NRMSE remain null when their denominators are undefined.

The summary averages window metrics within each participant and then weights participants equally; it is not a gait-cycle reproduction of the article tables. Check valid counts and individual windows alongside the means. Synthetic scores describe the demo only.

In [ ]:
if run_dir is not None:
    summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
    print("Run:", run_dir.name, "| units:", summary["units"], "| synthetic:", summary["synthetic_only"])
    for metric in ("mae", "rmse", "nrmse", "pcc", "r2", "ccc"):
        print(metric, summary[metric])
    from kinetics.plotting import plot_predictions
    figure = plot_predictions(run_dir / f"fold_{config.fold_ids[0]:02d}" / "predictions.npz")
    plt.show()
    plt.close(figure)
else:
    print("No new predictions: training was disabled.")

## 9. Reuse the saved model on source-scale windows

`FoldPredictor` loads the fold model and training transforms. It checks the saved model checksum (when available), window dimensions, sample rate and required channel names, then selects channels in their saved order and restores predictions to the source target scale. It never refits preprocessing and does not need reference targets.

Inputs must already be synchronized and use the training source's units and coordinate convention. The example takes the first held-out source window and compares its prediction with the saved export. For a new recording, supply aligned `(window,time,channel)` data and its actual channel names. Keep `model.keras`, `split.json` and the parent run's `config.json` together.

In [ ]:
if run_dir is not None:
    from kinetics.inference import FoldPredictor

    fold_dir = run_dir / f"fold_{config.fold_ids[0]:02d}"
    predictor = FoldPredictor(fold_dir)
    first_test = np.flatnonzero(np.isin(data.subject, split_manifest["subjects"]["test"]))[0]
    source_window = data.x[first_test:first_test + 1, :config.window_length, :]
    reloaded_prediction = predictor.predict_windows(source_window, data.feature_names, config.sample_rate)
    with np.load(fold_dir / "predictions.npz", allow_pickle=False) as archive:
        np.testing.assert_allclose(reloaded_prediction, archive["Y_pred"][:1], atol=1e-5, rtol=1e-5)
    print("Saved model reload matches the exported prediction.")
    del predictor
    tf.keras.backend.clear_session()
else:
    print("Reload check is available after a training run.")

## 10. Keep the experiment reproducible

Save the full run directory and the exact repository commit with any reported result. Keep data and generated runs outside version control. Historical exports under [outputs/](outputs/README.md) were not regenerated by this workflow.

For the next experiment, change one scientific factor at a time (target, foot, input channels, model or preprocessing), restart from Section 2, and retain a separate configuration. Consult the [paper correspondence and citation guide](docs/PAPERS.md) before comparing with either article. Research-data, GPU and article-level numerical validation remain separate from this synthetic demonstration.